# Modeling SNAP Policy Generosity Across U.S States 

##### By: By: Shawn Ding, Grace George, Razan Habboub, Arnav Jain, Aeon Levy
Click [Here](https://github.com/arnavjain321-prog/DS_6021_Final_Project) for Project GitHub

## 𝟙. Introduction & Project Overview

Our project studies how economic, demographic, and policy factors relate to the generosity of SNAP (Supplemental Nutrition Assistance Program) across U.S. states (including the D.C).

We used publicly available data from the U.S. Census Bureau, USDA, BLS, and state policy sources, we build a state-level dataset with:

- SNAP benefits and participation (May 2025)
- Poverty rates and median household income (ACS 2024)
- Food-at-home Regional Price Parities (RPPs) as a grocery cost index
- Racial composition shares
- Minimum wage levels and political trifecta control
- Rural–urban classification and SNAP administrative regions

We define a SNAP policy class for each state — Low, Moderate, or High — based on SNAP - participants per 1,000 residents. We then fit:

- A multinomial logistic regression model
- A random forest classifier

Both models aim to predict which states fall into each generosity class.

At a high level:

- The logistic regression achieves moderate performance and provides a transparent baseline.
- The random forest performs better overall, capturing nonlinear interactions between economics, demographics, and policy variables.
- Economic need (poverty, income), food prices, and policy choices (minimum wage, region, political control) are all associated with differences in SNAP generosity.
The goal of this analysis is not to prescribe “ideal” policy, but to describe how states cluster in terms of SNAP generosity and which observable factors help explain these differences.

### ✶ Reseach Question(s) and Motavations  ✶
SNAP is one of the main safety-net programs in the United States, but states differ substantially in how generous their SNAP benefits are in practice. Understanding what drives these differences can inform both policy discussion and future empirical work.


We focus on three main questions:

1. How can we construct a consistent, state-level measure of SNAP generosity?
    - We use SNAP participants per 1,000 residents and per-person benefit measures as core indicators of program reach and generosity.

2. How do economics, demographics, and policy environments relate to SNAP generosity?
    - We combine poverty, income, grocery cost indices, racial composition, minimum wage policy, political trifecta status, rural–urban mix, and USDA SNAP regions into a single modeling dataset.

3. Can we predict which states fall into Low, Moderate, or High SNAP generosity classes?
    - We compare a multinomial logistic regression baseline with a random forest classifier to see how well these observable factors classify states.


The analysis is cross-sectional and descriptive, so we do not claim causality. Our aim is to highlight patterns and associations.

### ✶ Selecting the Data ✶



We build a state-level dataset by merging the following sources:

- **ACS 2024 – Race (Table B02001):**  
  Total population and race counts, used to compute race shares by state.

- **ACS 2024 – Median Household Income (Table B19013):**  
  State-level median household income.

- **ACS 2024 – Poverty (Table S1701):**  
  Share of the population below the poverty line.

- **USDA SNAP Benefits & Participants (May 2025):**  
  State-level total SNAP benefits and number of participants.

- **BLS Regional Price Parities – Food-at-Home (RPP):**  
  State-level index of grocery cost (food-at-home price parity).

- **USDA SNAP Regions:**  
  Assignment of states to SNAP regional offices.

- **State Trifecta Control (2024):**  
  Whether each state is under Democratic, Republican, or divided trifecta control.

- **State Minimum Wage (2025):**  
  State-level minimum wage and a derived wage tier (low / medium / high).

- **Rural–Urban Continuum Codes (RUCC 2023):**  
  County-level RUCC codes aggregated to median RUCC by state and collapsed into 
  an urban / mixed / rural category.


## 𝟚. Engineering and Cleaning the Data


### ✶ Core Derived Variables ✶

From these raw sources, we constructed new varibles:

- **benefits_per_person** – May 2025 SNAP benefits divided by participants  
- **participants_per_1000** – SNAP participants per 1,000 residents  
- **grocery_cost_index** – 2023 food-at-home RPP value for each state  
- **poverty_rate** – ACS poverty share (proportion)  
- **median_household_income** – ACS median household income (USD)  
- **Race shares** – `white_pct`, `black_pct`, `native_pct`, `asian_pct`, 
  `pacific_pct`, `two_plus_pct`  
- **Policy and geography** – `min_wage_tier`, `trifecta_2024`, 
  `rural_urban_category`, `usda_snap_region`

The final modeling dataset has **51 rows (50 states + DC)** and a combined set of 
economic, demographic, and policy features for each state.




### ✶ Outcome: SNAP Policy Class ✶

We create a categorical target variable, **`snap_policy_class`**, that groups states 
into three SNAP generosity classes based on **SNAP participants per 1,000 residents**:

- **Low:** Below the 33rd percentile of `participants_per_1000`  
- **Moderate:** Between the 33rd and 66th percentiles  
- **High:** At or above the 66th percentile  

This construction ensures that we have a reasonably balanced three-class problem 
across the 51 states.

In [ ]:
# ============================================
# Imports and directory setup
# ============================================

from pathlib import Path

import numpy as np
import pandas as pd

from pandas.errors import ParserError

# For modeling and visualization
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns

# Base project directory = folder where this notebook lives
PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / "raw"
PROCESSED_DIR = PROJECT_DIR / "processed"

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("\nDirectory existence check:")
print("RAW_DIR exists:", RAW_DIR.exists())
print("PROCESSED_DIR exists:", PROCESSED_DIR.exists())


In [ ]:
# ============================================
# Load raw CSV data robustly
# ============================================

data_paths = {
    "acs_race":       RAW_DIR / "ACSDT1Y2024.B02001-Data.csv",
    "acs_income":     RAW_DIR / "ACSDT1Y2024.B19013-Data.csv",
    "acs_poverty":    RAW_DIR / "ACSST1Y2024.S1701-Data.csv",
    "snap_benefits":  RAW_DIR / "snap-benefits-8.csv",
    "snap_persons":   RAW_DIR / "snap-persons-8.csv",
    "state_trifecta": RAW_DIR / "state_trifecta_2024.csv",
    "snap_regions":   RAW_DIR / "usda_snap_regions.csv",
    "rpp":            RAW_DIR / "SARPP_STATE_2008_2023.csv",
    "rural_raw":      RAW_DIR / "Ruralurbancontinuumcodes2023.csv",
    "state_min_wage": RAW_DIR / "state_minimum_wage_2025.csv",
}

dfs = {}

for name, path in data_paths.items():
    try:
        df = pd.read_csv(path, dtype=str, encoding_errors="replace")
        sep_used = ","
    except ParserError:
        try:
            df = pd.read_csv(path, dtype=str, sep="\t", encoding_errors="replace")
            sep_used = "\\t (tab)"
        except Exception as e:
            print(f"!! {name:15s} FAILED to load from {path.name}: {e}")
            continue
    except FileNotFoundError:
        print(f"!! {name:15s} NOT FOUND at {path}")
        continue

    dfs[name] = df
    print(f"{name:15s} loaded from {path.name:35s} sep={sep_used} -> shape {df.shape}")

# Unpack for convenience
acs_race_raw   = dfs.get("acs_race")
acs_income_raw = dfs.get("acs_income")
acs_poverty_raw= dfs.get("acs_poverty")
snap_benefits_raw = dfs.get("snap_benefits")
snap_persons_raw  = dfs.get("snap_persons")
state_trifecta_raw= dfs.get("state_trifecta")
snap_regions_raw  = dfs.get("snap_regions")
rpp_raw           = dfs.get("rpp")
rural_raw         = dfs.get("rural_raw")
state_min_wage_raw= dfs.get("state_min_wage")


In [ ]:
# ============================================
# Inspect schemas and sample rows
# ============================================

def show_info(name, df, n_cols=8, n_rows=5):
    if df is None:
        print(f"\n{name}: DataFrame is None")
        return
    print("\n" + "="*80)
    print(f"{name} — shape: {df.shape}")
    print("- Columns:")
    print(list(df.columns))
    print("- Sample:")
    display(df.iloc[:n_rows, :n_cols])

show_info("acs_race_raw", acs_race_raw)
show_info("acs_income_raw", acs_income_raw)
show_info("acs_poverty_raw", acs_poverty_raw)
show_info("snap_benefits_raw", snap_benefits_raw)
show_info("snap_persons_raw", snap_persons_raw)
show_info("state_trifecta_raw", state_trifecta_raw)
show_info("snap_regions_raw", snap_regions_raw)
show_info("rpp_raw", rpp_raw)
show_info("rural_raw", rural_raw)
show_info("state_min_wage_raw", state_min_wage_raw)


In [ ]:
# ============================================
# Clean SNAP benefits and participants tables
# ============================================

def clean_snap_table(df, value_col_name):
    """
    SNAP tables have:
      - row 0: 'Data as of ...'
      - row 1: real header row (State / Territory, May 2024, April 2025, May 2025 Initial, etc.)
    This function:
      1) uses row 1 as header
      2) drops the first two rows
      3) extracts 'State / Territory' and 'May 2025 Initial'
      4) cleans numbers and renames the value column
    """
    df = df.copy()

    # Use row 1 as header
    header = df.iloc[1].tolist()
    df.columns = header

    # Drop first two rows
    df = df.iloc[2:].reset_index(drop=True)

    # Find state column
    state_col = [c for c in df.columns if "State / Territory" in str(c)][0]

    # Find the May 2025 Initial column (contains both pieces of text)
    may_cols = [c for c in df.columns if "May 2025" in str(c) and "Initial" in str(c)]
    may2025_col = may_cols[0]

    # Keep only needed columns
    df = df[[state_col, may2025_col]].rename(columns={state_col: "state"})

    # Clean numeric formatting
    df[may2025_col] = (
        df[may2025_col]
        .replace({"--": np.nan, "": np.nan})
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace("", np.nan)
        .astype(float)
    )

    # Rename value column
    df = df.rename(columns={may2025_col: value_col_name})

    return df

snap_benefits_clean = clean_snap_table(snap_benefits_raw, value_col_name="benefits_may_2025")
snap_persons_clean  = clean_snap_table(snap_persons_raw, value_col_name="participants_may_2025")

snap_benefits_clean.to_csv(PROCESSED_DIR / "snap_benefits_clean.csv", index=False)
snap_persons_clean.to_csv(PROCESSED_DIR / "snap_persons_clean.csv", index=False)

print("SNAP benefits cleaned ->", snap_benefits_clean.shape)
print("SNAP persons  cleaned ->", snap_persons_clean.shape)
snap_benefits_clean.head(), snap_persons_clean.head()


In [ ]:
# ============================================
# Clean ACS Race
# ============================================

def clean_acs_race(df):
    df = df.copy()

    # Keep only real states (ignore header row)
    df = df[df["GEO_ID"].str.startswith("0400000US")].reset_index(drop=True)

    # Extract state name
    df["state"] = df["NAME"]

    cols = {
        "total_pop": "B02001_001E",
        "white":     "B02001_002E",
        "black":     "B02001_003E",
        "native":    "B02001_004E",
        "asian":     "B02001_005E",
        "pacific":   "B02001_006E",
        "two_plus":  "B02001_008E",
    }

    for new_col, old_col in cols.items():
        df[new_col] = pd.to_numeric(df[old_col], errors="coerce")

    # Percentages
    for race in ["white", "black", "native", "asian", "pacific", "two_plus"]:
        df[f"{race}_pct"] = df[race] / df["total_pop"]

    keep_cols = ["state"] + [f"{race}_pct" for race in ["white", "black", "native", "asian", "pacific", "two_plus"]]
    df_clean = df[keep_cols].copy()

    return df_clean

acs_race_clean = clean_acs_race(acs_race_raw)
acs_race_clean.to_csv(PROCESSED_DIR / "acs_race_clean.csv", index=False)
print("acs_race_clean shape:", acs_race_clean.shape)
acs_race_clean.head()


In [ ]:
# ============================================
# Clean ACS Income
# ============================================

def clean_acs_income(df):
    df = df.copy()
    df = df[df["GEO_ID"].str.startswith("0400000US")].reset_index(drop=True)
    df["state"] = df["NAME"]
    df["median_household_income"] = pd.to_numeric(df["B19013_001E"], errors="coerce")
    df_clean = df[["state", "median_household_income"]].copy()
    return df_clean

acs_income_clean = clean_acs_income(acs_income_raw)
acs_income_clean.to_csv(PROCESSED_DIR / "acs_income_clean.csv", index=False)
print("acs_income_clean shape:", acs_income_clean.shape)
acs_income_clean.head()


In [ ]:
# ============================================
# Clean ACS Poverty
# ============================================

def clean_acs_poverty(df):
    df = df.copy()
    df = df[df["GEO_ID"].str.startswith("0400000US")].reset_index(drop=True)
    df["state"] = df["NAME"]
    poverty_col = "S1701_C03_001E"  # percent below poverty, estimate
    df["poverty_rate"] = pd.to_numeric(df[poverty_col], errors="coerce") / 100.0
    df_clean = df[["state", "poverty_rate"]].copy()
    return df_clean

acs_poverty_clean = clean_acs_poverty(acs_poverty_raw)
acs_poverty_clean.to_csv(PROCESSED_DIR / "acs_poverty_clean.csv", index=False)
print("acs_poverty_clean shape:", acs_poverty_clean.shape)
acs_poverty_clean.head()


In [ ]:
# ============================================
# Clean Food-at-Home RPP (Grocery Cost Index)
# ============================================

rpp = rpp_raw.copy()

# Keep only LineCode 7 (Food-at-home RPP)
food = rpp[rpp["LineCode"] == "7"].copy()

# Convert year columns to numeric values where possible
year_cols = [c for c in food.columns if c.isdigit()]
food[year_cols] = food[year_cols].apply(pd.to_numeric, errors="coerce")

# Extract 2023 Food-at-home RPP
food["grocery_cost_index"] = food["2023"]

# Clean state names: "Alabama, United States (STATE)" -> "Alabama"
food_clean = food[["GeoName", "grocery_cost_index"]].copy()
food_clean["state"] = food_clean["GeoName"].str.replace(", United States \\(STATE\\)", "", regex=True)

# Drop national rows or blanks
food_clean = food_clean[food_clean["state"].notna()]
food_clean = food_clean[~food_clean["state"].str.contains("United States")]

food_clean = food_clean[["state", "grocery_cost_index"]].copy()
food_clean.to_csv(PROCESSED_DIR / "grocery_cost_index_clean.csv", index=False)

print("grocery_cost_index_clean shape:", food_clean.shape)
food_clean.head()


In [ ]:
# ============================================
# Load state reference table (names, abbr, FIPS)
# ============================================

state_ref = pd.read_csv(PROCESSED_DIR / "state_reference.csv")
state_ref.head()


In [ ]:
# ============================================
# Build state-level master (without policy yet)
# ============================================

def clean_state_col(df, col="state"):
    df = df.copy()
    df[col] = df[col].str.strip()
    return df

acs_income      = clean_state_col(acs_income_clean, "state")
acs_poverty     = clean_state_col(acs_poverty_clean, "state")
acs_race        = clean_state_col(acs_race_clean, "state")
grocery_cost    = clean_state_col(food_clean, "state")
snap_benefits   = clean_state_col(snap_benefits_clean, "state")
snap_persons    = clean_state_col(snap_persons_clean, "state")
state_trifecta  = clean_state_col(state_trifecta_raw, "state")
snap_regions    = clean_state_col(snap_regions_raw, "state")

# Start from state_ref to ensure 50 states + DC
state_ref = state_ref.copy()
state_ref["state"] = state_ref["state_name"].str.strip()

master_df = state_ref[["state", "state_abbr", "fips_state"]].copy()

# Merge in all pieces
for df in [
    acs_poverty,
    acs_income,
    acs_race,
    grocery_cost,
    snap_benefits,
    snap_persons,
    state_trifecta,
    snap_regions,
]:
    master_df = master_df.merge(df, on="state", how="left")

# Calculate benefits per person
master_df["benefits_per_person"] = master_df["benefits_may_2025"] / master_df["participants_may_2025"]

master_df = master_df.sort_values("state").reset_index(drop=True)

print("master_df shape:", master_df.shape)
master_df.head()


In [ ]:
# ============================================
# Clean state minimum wage and RUCC
# ============================================

# Minimum wage
mw = state_min_wage_raw.copy()
mw.columns = [c.strip().lower().replace(" ", "_") for c in mw.columns]

if "minimum_wage_2025" in mw.columns:
    mw = mw.rename(columns={"minimum_wage_2025": "minimum_wage"})

mw["minimum_wage"] = (
    mw["minimum_wage"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

def categorize_wage(x):
    if x >= 14:
        return "high"
    elif x >= 10:
        return "medium"
    else:
        return "low"

mw["min_wage_tier"] = mw["minimum_wage"].apply(categorize_wage)

print("Cleaned Minimum Wage Data:")
mw.head()


In [ ]:
# RUCC: collapse county-level to state-level median
rural = rural_raw.copy()

rural_clean = rural[rural["Attribute"] == "RUCC_2023"].copy()
rural_clean["RUCC"] = rural_clean["Value"].astype(float).astype(int)

state_rucc = (
    rural_clean.groupby("State")["RUCC"]
    .median()
    .reset_index()
)

def classify_rucc(code):
    if code <= 3:
        return "urban"
    elif code <= 6:
        return "mixed"
    else:
        return "rural"

state_rucc["rural_urban_category"] = state_rucc["RUCC"].apply(classify_rucc)

# Drop territories like AS if present
state_rucc = state_rucc[state_rucc["State"] != "AS"].copy()
state_rucc["RUCC"] = state_rucc["RUCC"].astype(int)

state_rucc.head()


In [ ]:
# ============================================
# Merge policy variables into master
# ============================================

final_master = master_df.copy()

# Standardize keys
final_master["state_abbr"] = final_master["state_abbr"].astype(str).str.strip()
mw["state_abbr"]           = mw["state_abbr"].astype(str).str.strip()
state_rucc["State"]        = state_rucc["State"].astype(str).str.strip()

# Merge minimum wage
final_master = final_master.merge(
    mw[["state_abbr", "minimum_wage", "min_wage_tier"]],
    on="state_abbr",
    how="left"
)

# Merge RUCC
final_master = final_master.merge(
    state_rucc.rename(columns={"State": "state_abbr"})[["state_abbr", "RUCC", "rural_urban_category"]],
    on="state_abbr",
    how="left"
)

print("final_master shape:", final_master.shape)
final_master.head()


In [ ]:
# ============================================
# Add population and participants per 1,000 residents
# ============================================

# Extract population from ACS race table
acs_pop = acs_race_raw[[ "NAME", "B02001_001E" ]].copy()
acs_pop["state"] = acs_pop["NAME"]
acs_pop["population"] = pd.to_numeric(acs_pop["B02001_001E"], errors="coerce")
acs_pop = acs_pop[acs_pop["state"].notna()]
acs_pop = acs_pop[["state", "population"]]

final_master = final_master.merge(acs_pop, on="state", how="left")

final_master["participants_per_1000"] = (
    final_master["participants_may_2025"] / final_master["population"]
) * 1000

final_master[["state", "population", "participants_per_1000"]].head()


In [ ]:
# ============================================
# Define SNAP policy class (Low / Moderate / High)
# ============================================

q1 = final_master['participants_per_1000'].quantile(0.33)
q2 = final_master['participants_per_1000'].quantile(0.66)

print("33rd percentile:", q1)
print("66th percentile:", q2)

def classify_snap(x):
    if x < q1:
        return "Low"
    elif x < q2:
        return "Moderate"
    else:
        return "High"

final_master["snap_policy_class"] = final_master["participants_per_1000"].apply(classify_snap)
final_master["snap_policy_class"].value_counts()


## 𝟛. Data Visualization & Exploration

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier

import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv("final_master_dataset.csv")

# Quick check
print(df.shape)
print(df.head())
print(df.isna().sum())

# Drop the 1 missing trifecta value (tiny dataset so this is fine)
df = df.dropna(subset=["trifecta_2024"])


In [ ]:
# Columns we will NOT use as predictors
id_cols   = ["state", "state_abbr", "fips_state"]
leaky_cols = ["benefits_may_2025", "participants_may_2025"]  # directly define benefits/participation

target_reg = "benefits_per_person"
target_clf = "snap_policy_class"

num_features = [
    "poverty_rate",
    "median_household_income",
    "white_pct", "black_pct", "native_pct", "asian_pct", "pacific_pct", "two_plus_pct",
    "grocery_cost_index",
    "minimum_wage",
    "RUCC",
    "population",
    "participants_per_1000",
]

cat_features = [
    "trifecta_2024",
    "usda_snap_region",
    "min_wage_tier",
    "rural_urban_category",
]

# Just to be explicit:
X_all = df[num_features + cat_features]
y_reg = df[target_reg]
y_clf = df[target_clf]


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features),
    ]
)


- MAYBE EXPAIN WHY WE DID ABOVE CODE


### Simple Data Visualizations

Before staring  our analysis, we first explored the data visually to uncover patterns or trends. We created histograms to examine the distributions of key variables 


In the figure below we observed ...

- Poverty_rate is roughly normally distributed around 12%
- benefits_per_person is right-skewed, with most states between 
250
- participants_per_1000 is also right-skewed, indicating most states have lower participation
- minimum_wage is bimodal, reflecting the divide between states following federal vs. higher state minimum wages This shows substantial variation in key economic and policy variables across states.

In [ ]:
# ============================================
# Hist.1 Visualization: distribution of key variables
# ============================================
plt.figure(figsize=(12, 8))
df[["poverty_rate", "benefits_per_person", "grocery_cost_index",
    "participants_per_1000", "minimum_wage"]].hist(bins=10, figsize=(12, 8))
plt.tight_layout()
plt.show()



This histogram below shows how states are distributed in terms of SNAP participation intensity. 
- The dashed vertical lines show the 33rd and 66th percentile cutoffs
    -This what we defined as Low, Moderate, and High SNAP policy classes.

In [ ]:
# ============================================
# Hist.2 Visualization: distribution of SNAP participants per 1,000 residents
# ============================================


plt.figure(figsize=(8, 5))
sns.histplot(final_master["participants_per_1000"], bins=12, kde=False)
plt.axvline(q1, linestyle="--")
plt.axvline(q2, linestyle="--")
plt.xlabel("SNAP participants per 1,000 residents")
plt.ylabel("Number of states")
plt.title("Distribution of SNAP Participation Intensity across States")
plt.tight_layout()
plt.show()

In [ ]:
### HEAT MAP

plt.figure(figsize=(10, 8))
corr = df[num_features + [target_reg, "participants_per_1000"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap (Numeric Variables)")
plt.tight_layout()
plt.show()


Key Insights:
Predictors of SNAP Benefits:
benefits_per_person has strong positive correlations with asian_pct (0.62) and pacific_pct (0.75). This suggests states with larger Asian/Pacific Islander populations may have higher benefit levels (potentially due to cost-of-living or program eligibility differences).
benefits_per_person also correlates with median_household_income (0.35), likely because higher-income states have higher grocery costs (see grocery_cost_index vs. income correlation: 0.88).
Predictors of SNAP Participation:
participants_per_1000 correlates positively with poverty_rate (0.59) and negatively with median_household_income (-0.23)—intuitive, as lower-income/poverty-stricken states rely more on SNAP.
Multicollinearity Note:
asian_pct and pacific_pct are highly correlated (0.84), which may impact linear regression (addressed later with Ridge/Lasso regularization).

In [ ]:
#BOX PLOT 
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x="snap_policy_class", y="benefits_per_person")
plt.title("Benefits per Person by SNAP Policy Class")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x="snap_policy_class", y="participants_per_1000")
plt.title("Participants per 1000 by SNAP Policy Class")
plt.tight_layout()
plt.show()



BOXPLOTS INTERPRETATION:

"High" SNAP policy states provide significantly higher benefits per person
"High" policy states also have much higher participation rates per 1000 people This indicates that more generous SNAP policies involve both higher benefits AND broader program access. The policy classification effectively captures real differences in program implementation

In [ ]:
#SCATTER
plt.figure(figsize=(6,5))
sns.scatterplot(data=df,
                x="poverty_rate", y="participants_per_1000",
                hue="snap_policy_class", s=80)
plt.title("Poverty Rate vs Participants per 1000")
plt.tight_layout()
plt.show()


SCATTERPLOT INTERPRETATION:

Clear positive relationship: as poverty increases, SNAP participation increases.

"High" policy states (red) cluster in high-poverty, high-participation areas.

"Low" policy states (blue) are concentrated in lower-participation regions.

This visually confirms that need (poverty) drives participation, but policy choices moderate the response.

## Predicive Modeling Approaches



We use a mix of numeric and categorical predictors:

- **Numeric predictors:**
  - `benefits_per_person`
  - `participants_per_1000`
  - `poverty_rate`
  - `grocery_cost_index`
  - `median_household_income`
  - `white_pct`, `black_pct`, `native_pct`, `asian_pct`, `pacific_pct`, `two_plus_pct`

- **Categorical predictors:**
  - `min_wage_tier` (low / medium / high)
  - `trifecta_2024` (Democratic / Republican / Divided)
  - `rural_urban_category` (urban / mixed / rural)
  - `usda_snap_region` (USDA SNAP region labels)



We standardize numeric features and one-hot encode categorical features within a 
single `ColumnTransformer`, which is then used in all models for comparability.


In [ ]:
## MLP neural network (classification)
mlp_clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        max_iter=2000,
        random_state=42
    ))
])

mlp_clf.fit(X_train_clf, y_train_clf)

print("MLP accuracy:")
print("  Train:", mlp_clf.score(X_train_clf, y_train_clf))
print("  Test :", mlp_clf.score(X_test_clf, y_test_clf))

cv_scores_mlp = cross_val_score(mlp_clf, X_all, y_clf, cv=5)
print("CV accuracy (5-fold):", cv_scores_mlp.mean(), "+/-", cv_scores_mlp.std())







Then We split the data into **training** and **test** sets using a 70/30 split and preserve 
the class proportions with stratified sampling. All preprocessing (scaling numeric 
features and one-hot encoding categorical features) is handled inside a scikit-learn 
`Pipeline`.


In [ ]:
# ============================================
# Train/test split and preprocessing
# ============================================

# Target
y = final_master["snap_policy_class"]

# Features: drop identifiers and target
X = final_master.drop(columns=["snap_policy_class"])

categorical_cols = ["min_wage_tier", "trifecta_2024", "usda_snap_region", "rural_urban_category"]
numeric_cols = [
    "benefits_per_person", "participants_per_1000", "poverty_rate",
    "grocery_cost_index", "median_household_income",
    "white_pct", "black_pct", "native_pct",
    "asian_pct", "pacific_pct", "two_plus_pct"
]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)


### We compare two multi-class classification models:



1. **Multinomial Logistic Regression**  
   - Linear decision boundaries in the transformed feature space  
   - More interpretable coefficients, but limited in capturing complex interactions  
 

In [ ]:
# ============================================
# Multinomial Logistic Regression
# ============================================

log_reg_clf = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, multi_class="multinomial"))
])

log_reg_clf.fit(X_train, y_train)
y_pred_lr = log_reg_clf.predict(X_test)

print("Multinomial Logistic Regression Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))




The multinomial logistic regression serves as a transparent baseline model. On the 
held-out test set:

- Overall accuracy is about **69%**.  
- The **High** and **Low** classes are captured reasonably well (precision between
  about 0.75 and 1.00), while the **Moderate** class has higher recall but lower 
  precision, reflecting some confusion between neighboring categories.

This suggests that the linear model is able to pick up meaningful differences between 
Low, Moderate, and High SNAP generosity states, but the relationships are still not 
fully captured by simple linear decision boundaries in the feature space, especially 
for borderline cases between adjacent classes.


2. **Random Forest Classifier**  
   - Nonlinear, tree-based ensemble model  
   - Can capture interactions and nonlinearities at the cost of interpretability 

In [ ]:
# ============================================
# Random Forest Classifier
# ============================================

rf_clf = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=500, random_state=42))
])

rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)

print("Random Forest Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


In [ ]:
# ============================================
# Random Forest Feature Importances
# ============================================

rf_model = rf_clf.named_steps["clf"]

# One-hot encoded column names
ohe = preprocessor.named_transformers_["cat"]
ohe_feature_names = ohe.get_feature_names_out(categorical_cols)

feature_names = numeric_cols + list(ohe_feature_names)
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 8))
sns.barplot(
    y=np.array(feature_names)[indices],
    x=importances[indices]
)
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:

# # Extract trained RF model
# rf_model = rf_clf.named_steps["clf"]

# # One-hot encoded column names
# ohe = preprocessor.named_transformers_["cat"]
# ohe_feature_names = ohe.get_feature_names_out(categorical_cols)

# # Combine
# feature_names = numeric_cols + list(ohe_feature_names)

# # Importance values
# importances = rf_model.feature_importances_
# indices = np.argsort(importances)[::-1]

# plt.figure(figsize=(10,6))
# sns.barplot(
#     y=np.array(feature_names)[indices],
#     x=importances[indices],
#     palette="viridis"
# )
# plt.title("Random Forest Feature Importance")
# plt.xlabel("Importance Score")
# plt.ylabel("Feature")
# plt.tight_layout()
# plt.show()

The random forest uses 500 trees and the same preprocessed feature set. On the test set:

- Overall accuracy is notably higher than the logistic regression baseline.  
- Precision and recall are more balanced across the three classes (*Low*, *Moderate*, 
  *High*), indicating better separation between categories.

Because the random forest can model nonlinear relationships and interactions, it is 
better suited to capturing the complex way in which economic conditions, demographics, 
and policy environment jointly relate to SNAP generosity.


**Figure 2. Random Forest Feature Importances**

This figure highlights which variables are most influential in predicting a state’s 
SNAP policy class. Core economic measures (such as poverty, income, and benefits per 
person), food price levels, and policy variables (minimum wage tier, SNAP region, 
and political trifecta) typically appear among the top drivers, with demographic 
composition providing additional explanatory power.


### Logistic Regression (GLM) 

## DO I NEED BELOW? Arnav already trained above?? 

Need to add graces interp

In [ ]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_all, y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

log_reg_clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(
        multi_class="multinomial",
        max_iter=500,
        solver="lbfgs"
    ))
])

log_reg_clf.fit(X_train_clf, y_train_clf)

print("Logistic Regression (multinomial) accuracy:")
print("  Train:", log_reg_clf.score(X_train_clf, y_train_clf))
print("  Test :", log_reg_clf.score(X_test_clf, y_test_clf))


In [ ]:
# CROSS VALIDATION  of Multinomial Logistic Regression
cv_scores_log = cross_val_score(log_reg_clf, X_all, y_clf, cv=5)
print("CV accuracy (5-fold):", cv_scores_log.mean(), "+/-", cv_scores_log.std())


In [ ]:
## Regularization of Logistic Regression
### GRACES NEW CODE HERE

In [ ]:
# KNN
knn_clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

knn_clf.fit(X_train_clf, y_train_clf)

print("KNN accuracy:")
print("  Train:", knn_clf.score(X_train_clf, y_train_clf))
print("  Test :", knn_clf.score(X_test_clf, y_test_clf))

cv_scores_knn = cross_val_score(knn_clf, X_all, y_clf, cv=5)
print("CV accuracy (5-fold):", cv_scores_knn.mean(), "+/-", cv_scores_knn.std())


In [ ]:
### GRACE NEW CODE HERE

In [ ]:
## LIn reg

X_reg = df[num_features + cat_features]
y_reg = df[target_reg]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)


In [ ]:
#NEW GRACE 

In [ ]:
## PLAIN LINEAR REGRESSION
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

y_pred_train = linreg.predict(X_train_reg)
y_pred_test  = linreg.predict(X_test_reg)

train_rmse = np.sqrt(mean_squared_error(y_train_reg, y_pred_train))
test_rmse  = np.sqrt(mean_squared_error(y_test_reg,  y_pred_test))

print("Linear Regression:")
print("  Train RMSE:", train_rmse)
print("  Test  RMSE:", test_rmse)
print("  Train R^2:", r2_score(y_train_reg, y_pred_train))
print("  Test  R^2:", r2_score(y_test_reg,  y_pred_test))


In [ ]:
## RIDGE REGRESSION
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

ridge = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", Ridge(alpha=10.0))
])

ridge.fit(X_train_reg, y_train_reg)

y_pred_test_ridge = ridge.predict(X_test_reg)

ridge_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_test_ridge))
ridge_r2   = r2_score(y_test_reg, y_pred_test_ridge)

print("Ridge Regression:")
print("  Test RMSE:", ridge_rmse)
print("  Test R^2 :", ridge_r2)


In [ ]:
## LASSO REGRESSION

from sklearn.linear_model import Lasso

lasso = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", Lasso(alpha=1.0, max_iter=5000))
])

lasso.fit(X_train_reg, y_train_reg)

y_pred_test_lasso = lasso.predict(X_test_reg)

lasso_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_test_lasso))
lasso_r2   = r2_score(y_test_reg, y_pred_test_lasso)

print("Lasso Regression:")
print("  Test RMSE:", lasso_rmse)
print("  Test R^2 :", lasso_r2)


## 𝟜. Unsupervdied Learning 

In [ ]:
## K mean clustering (unsupoervised)
X_cluster = df[num_features]  # only numeric

scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_cluster_scaled)

df["cluster_k3"] = cluster_labels
print(df[["state", "cluster_k3", "snap_policy_class", "usda_snap_region"]].head())


In [ ]:
plt.figure(figsize=(6,5))
sns.scatterplot(
    x=df["poverty_rate"],
    y=df["participants_per_1000"],
    hue=df["cluster_k3"],
    palette="Set1",
    s=80
)
plt.title("K-means clusters (k=3) on poverty vs participants")
plt.tight_layout()
plt.show()


In [ ]:
# PCA dimentioniality reduction

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cluster_scaled)

print("Explained variance by PC1+PC2:", pca.explained_variance_ratio_.sum())

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "snap_policy_class": df["snap_policy_class"],
    "state": df["state"]
})

plt.figure(figsize=(7,6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2",
                hue="snap_policy_class", s=80)
for _, row in pca_df.iterrows():
    plt.text(row["PC1"]+0.02, row["PC2"]+0.02, row["state"][:2], fontsize=8)

plt.title("PCA (2D) of Numeric Features, colored by SNAP Policy Class")
plt.tight_layout()
plt.show()


In [ ]:
mlp_clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        max_iter=2000,
        random_state=42
    ))
])

mlp_clf.fit(X_train_clf, y_train_clf)

print("MLP accuracy:")
print("  Train:", mlp_clf.score(X_train_clf, y_train_clf))
print("  Test :", mlp_clf.score(X_test_clf, y_test_clf))

cv_scores_mlp = cross_val_score(mlp_clf, X_all, y_clf, cv=5)
print("CV accuracy (5-fold):", cv_scores_mlp.mean(), "+/-", cv_scores_mlp.std())

Need to add my interp 

## 𝟝. App/Dashboard 

In [ ]:
#RAZAN NEW APP CDOE

## 𝟞. Conclusion and Policy Takeaways

In this project, we constructed a state-level view of SNAP generosity and modeled how 
economic, demographic, and policy factors relate to whether a state falls into a 
Low, Moderate, or High SNAP policy class.

**Key takeaways:**

- States with **higher SNAP participation per 1,000 residents** tend to share a mix of 
  higher economic need (higher poverty or lower income), policy decisions that support 
  higher benefit levels, and, in some cases, higher grocery costs that may necessitate 
  more generous support.

- **Policy levers matter.** Minimum wage tiers, political trifecta control, and SNAP 
  administrative regions show meaningful associations with the SNAP policy classes in 
  the random forest model. This suggests that institutional and political context helps 
  shape how generous SNAP ultimately is in practice.

- **Demographic composition contributes but is not the whole story.** Race shares help 
  explain some variation, but economic and policy factors remain important even after 
  controlling for demographics.

The analysis is cross-sectional and descriptive, so we cannot infer causality. However, 
it provides a compact framework for thinking about SNAP generosity as the outcome of 
interacting economic conditions and policy choices.

Future work could extend this by using panel data over time, adding state-level 
unemployment and housing cost measures, or exploring causal designs to separate 
policy effects from underlying economic need.


𝟙𝟚𝟛𝟜𝟝𝟞𝟟𝟠𝟡𝟘